### <font color='blue'> PROJETO: AUTOMATIZAÇÃO DE ARQUIVOS EXCEL</font>

In [45]:
# Bibliotecas
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown
#!pip install xlsxwriter

In [46]:
display(Markdown("""
#### Resumo do projeto:

O código consolida planilhas de Excel distribuídas em várias pastas, gerando um arquivo único com todas as informações, além de uma aba com o resumo detalhado do faturamento e das vendas mensais por produto.

**O código foi dividido em 5 etapas:**

- **1.** Listando os arquivos da pasta VENDAS (pasta principal);
- **2.** Processando os dados;
- **3.** Gerando as tabelas;
- **4.** Cria uma cópia do arquivo na pasta HISTORICO ARQUIVOS;
- **5.** Renomeia o arquivo.

"""))


#### Resumo do projeto:

O código consolida planilhas de Excel distribuídas em várias pastas, gerando um arquivo único com todas as informações, além de uma aba com o resumo detalhado do faturamento e das vendas mensais por produto.

**O código foi dividido em 5 etapas:**

- **1.** Listando os arquivos da pasta VENDAS (pasta principal);
- **2.** Processando os dados;
- **3.** Gerando as tabelas;
- **4.** Cria uma cópia do arquivo na pasta HISTORICO ARQUIVOS;
- **5.** Renomeia o arquivo.



#### <font color='blue'> **1.** LISTANDO OS ARQUIVOS </font>

In [47]:
display(Markdown("""
O código busca todos os arquivos **Excel** que estão dentro da pasta principal ***'VENDAS'***, e das demais pastas, ***'JANEIRO'***, ***'FEVEREIRO'*** e 
***'MARÇO'***.
"""))


O código busca todos os arquivos **Excel** que estão dentro da pasta principal ***'VENDAS'***, e das demais pastas, ***'JANEIRO'***, ***'FEVEREIRO'*** e 
***'MARÇO'***.


In [48]:
# Pasta diretorio
pasta_raiz = Path('VENDAS')
lista_dataframes = []

# 1. Coleta e consolidação dos dados

for arquivo in pasta_raiz.rglob('*.xlsx'):
    try:
        dict_abas = pd.read_excel(arquivo, sheet_name=None)   # sheet_name=None carrega todas as abas em um dicionário
        for df in dict_abas.values():
            if not df.empty:                                              
                df['Mes'] = arquivo.parent.name               # O nome do mês é extraído da pasta onde o arquivo está                         
                lista_dataframes.append(df)
    except Exception as e:
        print(f"Erro ao ler {arquivo.name}: {e}")

Erro ao ler ~$Janeiro.xlsx: [Errno 13] Permission denied: 'VENDAS\\1 JANEIRO\\~$Janeiro.xlsx'


#### <font color='blue'> **2.** PROCESSANDO OS DADOS </font>

#### <font color='blue'> **3.** GERANDO AS TABELAS </font>

In [49]:
display(Markdown("""
#### 2. Processando os dados
Os dados de todos os arquivos **Excel** são consolidados em um novo arquivo denominado **Arquivos Consolidados**.

#### 3. Gerando as tabelas
O arquivo **Arquivos Consolidados** será estruturado em duas abas principais: a **"Dados Consolidados"**, contendo o banco de dados bruto, e a **"Resumo Mensal"**. Esta última apresentará visões analíticas detalhadas por meio das tabelas de ***'ANÁLISE MENSAL'***, ***'ANÁLISE DE VENDAS'*** e ***'ANÁLISE DE FATURAMENTO'***."

"""))


#### 2. Processando os dados
Os dados de todos os arquivos **Excel** são consolidados em um novo arquivo denominado **Arquivos Consolidados**.

#### 3. Gerando as tabelas
O arquivo **Arquivos Consolidados** será estruturado em duas abas principais: a **"Dados Consolidados"**, contendo o banco de dados bruto, e a **"Resumo Mensal"**. Esta última apresentará visões analíticas detalhadas por meio das tabelas de ***'ANÁLISE MENSAL'***, ***'ANÁLISE DE VENDAS'*** e ***'ANÁLISE DE FATURAMENTO'***."



In [50]:
# O processamento só deve ocorrer se houver arquivos na lista

# --- Consolidação inicial
if lista_dataframes:
    df_consolidado = pd.concat(lista_dataframes, ignore_index=True)
    
# --- FORMATANDO A COLUNA DATA ---    
    if 'Data' in df_consolidado.columns:                                
        df_consolidado['Data'] = pd.to_datetime(df_consolidado['Data'])

# --- EXPORTAÇÃO ---
    # Removemos a coluna auxiliar 'Mes' para a aba de dados brutos
    df_saida_bruta = df_consolidado.drop(columns=['Mes'])
        
# --- PROCESSAMENTO DAS TABELAS (Resumos) ---
    # Tabela 1: Resumo Mensal Consolidado (Vendas e Faturamento Total)
    resumo_mensal = df_consolidado.groupby('Mes').agg({ 
        'Vendas': 'sum',
        'Faturamento': 'sum'
    }).reset_index()

    # Adicionar uma linha de TOTAL ao final do resumo_mensal
    total_vendas = resumo_mensal['Vendas'].sum()
    total_fat = resumo_mensal['Faturamento'].sum()
    
    # Criando a linha de total formatada
    linha_total = pd.DataFrame({
        'Mes': ['TOTAL'], 
        'Vendas': [total_vendas], 
        'Faturamento': [total_fat]
    })
    
    resumo_mensal = pd.concat([resumo_mensal, linha_total], ignore_index=True)

    
    # Tabela 2: Analise mensal de vendas por produto
    pivot_vendas = pd.pivot_table(
        df_consolidado, values='Vendas', index='Produto', 
        columns='Mes', aggfunc='sum', fill_value=0, 
        margins=True, margins_name='TOTAL'
    ).reset_index()

    # Tabela 3: Analise mensal de faturamento por produto
    pivot_faturamento = pd.pivot_table(
        df_consolidado, values='Faturamento', index='Produto', 
        columns='Mes', aggfunc='sum', fill_value=0, 
        margins=True, margins_name='TOTAL'
    ).reset_index()
    
    # ExcelWriter é para salvar em varios sheets
with pd.ExcelWriter('Arquivos_Consolidados.xlsx', 
                        engine='xlsxwriter', 
                        datetime_format='dd/mm/yyyy') as writer:
        
        # --- 1. Aba de Dados Consolidados (BD_DADOS) ---
        df_saida_bruta.to_excel(writer, sheet_name='BD_DADOS', index=False)
        worksheet_dados = writer.sheets['BD_DADOS']
        
        # Loop para ajustar todas as colunas da aba Dados_Consolidados
        for i, col in enumerate(df_saida_bruta.columns):
            column_len = max(df_saida_bruta[col].astype(str).str.len().max(), len(col))+2 #Encont o > comprim entre o nome da col e os dados das céls
            worksheet_dados.set_column(i, i, column_len)

        # --- 2. Aba de Resumo Mensal (RESUMO) ---
        nome_aba = 'RESUMO'
        
        # Títulos das tabelas
        pd.Series(['ANÁLISE MENSAL DE VENDAS E FATURAMENTO']).to_excel(writer, sheet_name=nome_aba, index=False, header=False, startrow=0, startcol=0)
        pd.Series(['ANALISE MENSAL DE VENDAS POR PRODUTO']).to_excel(writer, sheet_name=nome_aba, index=False, header=False, startrow=0, startcol=5)
        pd.Series(['ANÁLISE MENSAL DE FATURAMENTO POR PRODUTO']).to_excel(writer,sheet_name=nome_aba,index=False,header=False,startrow=0, startcol=12)
        
        # Gerando as Tabelas
        resumo_mensal.to_excel(writer, sheet_name=nome_aba, index=False, startrow=2)
        pivot_vendas.to_excel(writer, sheet_name=nome_aba, index=False, startrow=2, startcol=5)
        pivot_faturamento.to_excel(writer, sheet_name=nome_aba, index=False, startrow=2, startcol=12)

        # Ajuste das colunas na aba 'RESUMO'
        worksheet_resumo = writer.sheets[nome_aba]
        worksheet_resumo.set_column('A:Z', 18) # Define uma largura das colunas nas tabelas

#### <font color='blue'> **4.** CRIANDO CÓPIA DO ARQUIVO 'Arquivos Consolidados' </font>

In [51]:
display(Markdown("""
O código gera uma cópia do arquivo **Arquivos Consolidados** na pasta ***HISTORICO ARQUIVOS***.

"""))


O código gera uma cópia do arquivo **Arquivos Consolidados** na pasta ***HISTORICO ARQUIVOS***.



In [52]:
import shutil
shutil.copy2('Arquivos_Consolidados.xlsx', 'HISTORICO ARQUIVOS\Arquivos_Consolidados.xlsx') #Criando cópia do arq 'Arqs_Consols' na pasta 'Historico'

'HISTORICO ARQUIVOS\\Arquivos_Consolidados.xlsx'

#### <font color='blue'> **5.** RENOMEANDO OS ARQUIVOS </font>

In [53]:
display(Markdown("""
O código renomeia o novo arquivo gerado com o nome do **(Arquivos_Consolidados)** com data e hora atual.
"""))


O código renomeia o novo arquivo gerado com o nome do **(Arquivos_Consolidados)** com data e hora atual.


In [54]:
import os
from datetime import datetime
import glob

# Configurações: Pasta e arquivo
pasta = 'HISTORICO ARQUIVOS' 
arquivo_antigo = "Arquivos_Consolidados.xlsx"
caminho_completo_antigo = os.path.join(pasta, arquivo_antigo)

# Obter data e hora atual formatada (AnoMesDia-HoraMinuto)
agora = datetime.now().strftime("%d%m%Y_%H%M")

# Criar o novo nome do arquivo
nome, extensao = os.path.splitext(arquivo_antigo)
novo_arquivo = f"{nome}_{agora}{extensao}"
caminho_completo_novo = os.path.join(pasta, novo_arquivo)

# Renomear o arquivo
try:
    os.rename(caminho_completo_antigo, caminho_completo_novo)
    print(f"Arquivo renomeado com sucesso para: {novo_arquivo}")
except FileNotFoundError:
    print("O arquivo original não foi encontrado.")
except FileExistsError:
    print("Já existe um arquivo com esse novo nome.")
except Exception as e:
    print(f"Ocorreu um erro: {e}")

Arquivo renomeado com sucesso para: Arquivos_Consolidados_22042026_1606.xlsx


##### <font color='blue'> Exclui o 1º arquivo 'Arquivos Consolidados' gerado na pasta raiz. </font>

In [55]:
# Excluindo arquivo gerado
os.remove('Arquivos_Consolidados.xlsx')